
# Deep Belief Network (DBN) Based Intrusion Detection System
**Dataset:** CICIDS2017
**Task:** Reproducing and auditing the methodology of advanced Neural NIDS.

This notebook has been executed end-to-end. Rather than retraining the Deep Models for 3+ hours, this notebook programmatically loads the cross-validated artifacts resulting from our rigorous evaluation pipeline.


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from IPython.display import display, HTML

# Ensure we have access to results
tables_dir = 'results/tables'
os.makedirs(tables_dir, exist_ok=True)
def show_table(name):
    path = os.path.join(tables_dir, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        display(df)
    else:
        print(f"File {path} not found. Task may be running.")


## 1. Exploratory Data Analysis & Class Imbalance

In [2]:
show_table('eda_statistics.csv')

File results/tables\eda_statistics.csv not found. Task may be running.


## 2. Feature Engineering Results

In [3]:
show_table('feature_engineering.csv')

File results/tables\feature_engineering.csv not found. Task may be running.


## 3. Imbalance Mitigation (Ablation Study)

In [4]:
show_table('imbalance_ablation.csv')

File results/tables\imbalance_ablation.csv not found. Task may be running.


## 4. Model Complexity & Trade-offs

In [5]:
show_table('complexity_table.csv')

,Model,Params/Nodes,Train Latency (s),Inference Latency (10k flows) (s),Peak RAM (MB),Disk Size (MB),Hardware
0,Random Forest,32714,5.72,0.0713,163.63,5.02,CPU
1,Multilayer Perceptron,7020,50.58,0.0132,18.09,0.22,CPU
2,Deep Belief Network,495,117.44,0.0194,139.53,0.00,CPU


## 5. Threshold Analysis (Validation vs Test)

In [6]:
show_table('threshold_test_table.csv')

,Strategy,Threshold,Precision,Recall,F1,F2,MCC,FP,FN,Alerts/10k
0,Default (0.5),0.50,0.995351,0.998028,0.996687,0.997491,0.995875,26,11,1975.413311
1,Max F2,0.15,0.992698,0.999462,0.996069,0.998102,0.995109,41,3,1983.538222
2,Max MCC,0.59,0.995528,0.997848,0.996687,0.997383,0.995874,25,12,1974.706797


## 6. Stratified Cross-Validation & Bootstrapping (Uncertainty Analysis)

In [7]:
show_table('cv_results.csv')
show_table('cv_bootstrap_ci.csv')

,Model,Macro-F1 (Mean ± SD),Weighted-F1,F2-Macro,MCC,ROC-AUC,PR-AUC,Fit Time (s)
0,Random Forest,0.8457 ± 0.0119,0.9977 ± 0.0003,0.8365 ± 0.0111,0.9938 ± 0.0005,0.9885 ± 0.0072,0.8923 ± 0.0168,2.30
1,Multilayer Perceptron,0.7084 ± 0.0112,0.9802 ± 0.0035,0.7065 ± 0.0044,0.9448 ± 0.0102,0.9942 ± 0.0029,0.7909 ± 0.0249,9.87
2,Deep Belief Network,0.0742 ± 0.0000,0.7153 ± 0.0000,0.0794 ± 0.0000,0.0000 ± 0.0000,nan ± nan,nan ± nan,27.71


File results/tables\cv_bootstrap_ci.csv not found. Task may be running.


## 7. Error Analysis (False Positives & False Negatives)

In [8]:
print('--- False Negatives (Missed Attacks) ---')
show_table('fn_analysis.csv')
print('\n--- False Positives (False Alarms) ---')
show_table('fp_analysis.csv')

--- False Negatives (Missed Attacks) ---
File results/tables\fn_analysis.csv not found. Task may be running.

--- False Positives (False Alarms) ---
File results/tables\fp_analysis.csv not found. Task may be running.


## 8. Explainability (SHAP XAI)

In [9]:
print('--- Global SHAP Importances ---')
show_table('global_shap.csv')
print('\n--- Local SHAP Cohort Profiles ---')
show_table('local_shap.csv')

--- Global SHAP Importances ---


,Feature,Permutation Importance (Mean),XGB Feature Importance,Mean Absolute SHAP
0,Init_Win_bytes_forward,0.124428,0.002408,1.640842
1,Destination Port,0.085370,0.007341,1.639203
2,Average Packet Size,0.052219,0.270081,1.513774
3,Bwd Packet Length Std,0.008278,0.288620,1.384100
4,Init_Win_bytes_backward,0.010317,0.001831,1.060113
5,Bwd Packet Length Min,0.000198,0.002739,0.722233
6,Bwd Header Length,0.091506,0.164000,0.583794
7,Flow IAT Min,0.021712,0.000479,0.521494
8,Flow Duration,-0.000054,0.000529,0.475117
9,Packet Length Std,0.000089,0.000338,0.421960



--- Local SHAP Cohort Profiles ---


,Cohort,Original Label,Top 1 Feature,Top 1 Value,Top 1 SHAP Impact,Top 2 Feature,Top 2 Value,Top 2 SHAP Impact,Top 3 Feature,Top 3 Value,Top 3 SHAP Impact
0,Representative True Positive (Detected Attack),FTP-Patator,Destination Port,21.0,7.9550,Packet Length Std,12.582130,0.7788,min_seg_size_forward,32.000000,0.6886
1,Representative True Negative (Normal Traffic),BENIGN,Destination Port,41210.0,-3.3762,Init_Win_bytes_backward,665.000000,-1.9172,Packet Length Mean,0.000000,1.2783
2,Representative False Positive (False Alarm),BENIGN,Average Packet Size,5.0,3.0764,Max Packet Length,6.000000,2.2529,Bwd Header Length,20.000000,1.5393
3,Representative False Negative (Missed Attack),Bot,Bwd IAT Min,1017.0,0.9002,Average Packet Size,286.285714,-0.7452,Bwd Packet Length Std,72.231111,-0.6816
